# EEG Reader
## What does this notebook do?

1. Taking in an .xdf file and reading its values. (including getting the files from another source)
2. Display the signal data including its markers visually.
3. Format the data correctly for a common table/dataset approach, including
  - Removing the 5s pause in between
  - Assign the respective snippets to their genre
  - Create a folder for that, specifying the session number this was
  - (Maybe) convert to MNE format for easier preprocessing

In [ ]:
# Important constants
SESSION_NUMBER = 0

## Example usage from https://github.com/xdf-modules/pyxdf

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pyxdf

# Test
data, header = pyxdf.load_xdf("sub-P001_ses-S001_task-Default_run-001_eeg.xdf")

for stream in data:
    y = stream["time_series"]

    if isinstance(y, list):
        # list of strings, draw one vertical line for each marker
        for timestamp, marker in zip(stream["time_stamps"], y):
            plt.axvline(x=timestamp)
            print(f'Marker "{marker[0]}" @ {timestamp}s')
    elif isinstance(y, np.ndarray):
        # numeric data, draw as lines
        plt.plot(stream["time_stamps"], y)
    else:
        raise RuntimeError("Unknown stream format")

plt.show()

In [ ]:
import pyxdf
import mne
import matplotlib.pyplot as plt
import numpy as np

# Load the XDF file
xdf_file = "sub-P001_ses-S001_task-Default_run-001_eeg.xdf"  # Replace with your file path
streams, file_header = pyxdf.load_xdf(xdf_file)

# Find EEG stream and marker stream
eeg_stream = next((s for s in streams if s['info']['type'][0] == 'EEG'), None)
marker_stream = next((s for s in streams if s['info']['type'][0] in ['Markers', 'Trigger']), None)

if eeg_stream is None:
    raise ValueError("No EEG stream found.")
if marker_stream is None:
    print("No marker stream found, proceeding without markers.")

# Extract EEG data
eeg_data = eeg_stream['time_series'].T  # Transpose: channels x samples
eeg_times = eeg_stream['time_stamps']   # Timestamps

# Channel info
ch_names = [ch['label'][0] for ch in eeg_stream['info']['desc'][0]['channels'][0]['channel']]
sfreq = float(eeg_stream['info']['nominal_srate'][0])

# Create MNE Raw object
info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
raw = mne.io.RawArray(eeg_data, info)
# raw.set_montage('standard_1020')  # Set standard electrode positions

# Plot EEG signals
fig = raw.plot(scalings='auto', show=True, title='EEG Signals', duration=10)

# Overlay markers
if marker_stream:
    marker_times = marker_stream['time_stamps']
    marker_values = [m[0] for m in marker_stream['time_series']]
    
    # Create Annotations
    onset = np.array(marker_times) - raw.first_time  # Align to raw time base
    duration = np.zeros_like(onset)
    raw.set_annotations(mne.Annotations(onset=onset, duration=duration, description=marker_values))

    print(f"{len(marker_values)} markers loaded and displayed on plot.")

plt.show()
